# Preparation

In [ ]:
%pip install polars
%pip install altair


In [222]:
import polars as pl
import os
from dataclasses import dataclass, field
from typing import Optional
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [223]:
os.chdir("/home/tsetsi/Python/financial-data-science-jupyter")
WORK_DIR = os.getcwd()
DATA = os.path.join(WORK_DIR, "data")

In [224]:
files = {
    "quotes_inc_eu": "DE0007500001_quotes_incremental.csv",
    "quotes_inc_us": "US2561631068_quotes_incremental.csv",
    "trades_eu": "DE0007500001_trades.csv",
    "trades_us": "US2561631068_trades.csv"
}

In [225]:
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64,
    "best_bid_price": pl.Float64,
    "best_ask_price": pl.Float64,
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [226]:
@dataclass
class LimitOrderBook:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","

    def __post_init__(self):
        # pl.scan_csv doesn't load the file into memory immediately but only when called with the
        # .collect() method, which generally makes running the code significantly faster.
        # schema_overrides is optional and can be used to explicitly set a data type to a column,
        # but it will return an error if polars finds some kind of mismatch.
        # def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

@dataclass
class Trades:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","
    
    def __post_init__(self):
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

In [227]:
## EU: Incremental quotes
quotes_inc_eu = LimitOrderBook(
    file=files["quotes_inc_eu"],
    folder_path=DATA,
    schema_override=quotes_inc_eu_schema
)

quotes_inc_eu = (
    quotes_inc_eu.df.sort(by=[pl.col("original_order_id"), 
                              pl.col("event_timestamp")]
    )
)

In [228]:
# df.sort(["group_id", "timestamp"]).with_columns(
#     pl.col("value")
#     .diff()
#     .over("group_id")
#     .fill_null(0)
#     .cum_sum()
#     .over("group_id")
# )

quotes_inc_eu = quotes_inc_eu. \
    sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .with_columns(
        pl.when(pl.col("trade_id") == 0) \
            .then(None) \
            .otherwise(pl.col("trade_id")) \
            .alias("trade_id"),
        pl.col("size") \
            .diff() \
            .over("original_order_id")
            .fill_null(pl.col("size"))
            .alias("size_diff"),
        pl.col("event_timestamp") \
            .rank(method="ordinal") \
            .over("original_order_id") \
            .alias("order_order"),
        pl.col("size") \
            .diff() \
            .over("original_order_id") \
            .fill_null(0) \
            .cum_sum() \
            .over("original_order_id")
            .alias("size_diff_cum_sum")
    )

In [229]:
# Sanity check: No original_order_id's exist within more than one venue.
print(
    quotes_inc_eu.group_by("original_order_id") \
        .agg([
            pl.col("venue").unique().alias("venues")
        ])
        .filter(pl.col("venues").list.len() > 1)
        .collect()
)

shape: (0, 2)
┌───────────────────┬───────────┐
│ original_order_id ┆ venues    │
│ ---               ┆ ---       │
│ i64               ┆ list[str] │
╞═══════════════════╪═══════════╡
└───────────────────┴───────────┘


In [448]:
def show_histogram(df: pl.LazyFrame, column_name: str, x_label: str = None, y_label: str = None, sort_order: list | str = None) -> None:
    
    x=alt.X(
        f"{column_name}:N", 
        sort=alt.SortField(sort_order if sort_order is not None else alt.Undefined), 
        axis=alt.Axis(labelAngle=0, title=x_label or column_name)
    )
    
    (
        alt.layer(
            alt.Chart(df.collect()).mark_bar().encode(
                x=x,
                y="count():Q",
            ),
            alt.Chart(df.collect()).mark_text(dy=-5).encode(
                x=x,
                y="count():Q",
                text="count():Q",
            ),
        )
        .properties(width=750, height=250)
    ).display()

In [449]:
show_histogram(
    df=quotes_inc_eu, 
    column_name="venue",
    x_label="Venues",
    sort_order="venue"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="market_state",
    x_label="Market states",
    sort_order="market_state"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="lob_action",
    x_label="LOB Actions",
    sort_order="lob_action"
)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

In [232]:
# Sanity check: Every order should have a REMOVE operation at some point
# and at some market state
orders_without_remove = (
    quotes_inc_eu.group_by(pl.col("original_order_id"))
    .agg(pl.col("lob_action").unique().alias("lob_actions"))
    .filter(
        # pl.col("lob_actions").list.contains("INSERT") &
        pl.col("lob_actions").list.contains("REMOVE").not_()
    )
)

orders_without_remove.collect().show(limit=20)

original_order_id,lob_actions
i64,list[str]


In [444]:
# Ideally, all trade_id's in the LOB would be found in the trades dataset,
# but this does not seem to be the case:
# I take all unique trade_id's in the LOB and I compare them against the trades dataset.
# Interestingly, the relevant trade_ids are associated only aggressor_side = UNKNOWN.
# Changing to from anti to inner join in trade_id_check will return BID and ASK only.
trade_ids_unique = quotes_inc_eu.select(pl.col("trade_id")).unique()

trade_id_check = trades_eu.join(
    trade_ids_unique,
    on="trade_id",
    how="anti"
)

show_histogram(
    df=trade_id_check, 
    column_name="trade_type",
    x_label="Trade Types",
    sort_order="trade_type"
)

show_histogram(
    df=trade_id_check, 
    column_name="aggressor_side",
    x_label="Aggressor Side",
    sort_order="aggressor_side"
)

trade_id_check.collect().sample(10)

alt.LayerChart(...)

alt.LayerChart(...)

trade_id,trade_timestamp,publication_timestamp,aggressor_side,price,execution_size,market_state,trade_type,venue
i128,str,str,str,f64,i64,str,str,str
42943562053403,"""2023-09-01 13:38:26.831064000""","""2023-09-01 13:38:26.831064000""","""UNKNOWN""",7.408,142,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943561960149,"""2023-09-01 08:14:16.450090000""","""2023-09-01 08:14:16.450090000""","""UNKNOWN""",7.288,74,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943561960862,"""2023-09-01 08:16:21.448247000""","""2023-09-01 08:16:21.448247000""","""UNKNOWN""",7.281,529,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943561956435,"""2023-09-01 08:02:57.043440000""","""2023-09-01 08:02:57.043440000""","""UNKNOWN""",7.282,631,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562001832,"""2023-09-01 10:47:29.489553000""","""2023-09-01 10:47:29.489553000""","""UNKNOWN""",7.304,1298,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
211963,"""2023-09-01 15:35:07.465751000""","""2023-09-01 15:35:07.465751000""","""UNKNOWN""",7.37,619,"""POST_TRADE""","""CLOSING_PRICE""","""AQEU"""
42943562099467,"""2023-09-01 15:09:06.638083000""","""2023-09-01 15:09:06.638083000""","""UNKNOWN""",7.382,120,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562067023,"""2023-09-01 14:01:01.209184000""","""2023-09-01 14:01:01.209184000""","""UNKNOWN""",7.364,174,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943561952540,"""2023-09-01 07:49:02.653563000""","""2023-09-01 07:49:02.653563000""","""UNKNOWN""",7.302,452,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""


# 3.1.

In [398]:
# Here I'm preparing the one-row-per-order dataset from the LOB.
# quotes_aggs 
quotes_aggs = [
    pl.col("venue") \
        .first()
        .alias("venue"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_date"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("removal_date"),
    pl.col("event_timestamp") \
        .max()
        .alias("latest_event_timestamp"),
    pl.col("lob_action") \
        .eq("UPDATE")
        .sum()
        .alias("number_of_updates"),
    pl.col("price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("price_at_insertion"),
    pl.col("size") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("size_at_insertion"),
    pl.col("old_price") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("old_price_at_removal"),
    pl.col("old_size") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("old_size_at_removal"),
    # EXECUTION SIZE
    # This sums the execution sizes of all original_order_id's
    # in preparation for determining the removal mechanism.
    pl.col("execution_size") \
        .filter(pl.col("order_executed")==True)
        .sum() # .first()
        .alias("execution_size"),
    pl.col("price_level") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_level"),
    pl.col("best_bid_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .fill_null(0)
        .first()
        .alias("best_bid_price_at_insertion"),
    pl.col("best_ask_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_ask_price_at_insertion"),
    pl.struct(
        [
            pl.col("event_timestamp") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("lob_updates"),
            pl.col("price") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_prices"),
            pl.col("size") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_sizes"),
        ]
    ),
    pl.struct(
        [
            pl.col("trade_id") \
                .filter((pl.col("trade_id").is_not_null()) | 
                        (pl.col("trade_id") != 0)) \
                .unique() \
                .alias("associated_trade_ids")
        ]
    ),
    pl.col("market_state") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("market_state_at_removal"),
    pl.col("size_diff") \
        .filter((pl.col("lob_action")!="REMOVE") & 
                (pl.col("order_executed")==False)) \
        .sum()
        .alias("size_diff_tally"),
]

quotes_calc = [
    (
        pl.when(pl.col("removal_date").is_not_null())
            .then(pl.col("removal_date"))
            .otherwise(pl.col("latest_event_timestamp"))
        - pl.col("insertion_date")
    ).dt.total_microseconds().alias("order_lifetime"),

    # (best bid price + best ask price) / 2
    (
        (
            pl.col("best_bid_price_at_insertion")
            + pl.col("best_ask_price_at_insertion")
        ) / 2
    ).alias("midpoint_at_insertion"),
]

In [452]:
quotes_collapsed = (
    quotes_inc_eu
    .group_by("original_order_id")
    .agg(quotes_aggs)
    .with_columns(quotes_calc)
)

log_order_lifetime_bins = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 30]
log_order_lifetime_bin_labels = ['00-02', '02-04', '04-06', '06-08', '08-10', '10-12', '12-14', '14-16', '16-18', '18-20', '20-22', '22-24', '24-30', '>=30']

quotes_collapsed = quotes_collapsed.with_columns([
    # Absolute distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    (
        pl.col("price_at_insertion")
        .sub(pl.col("midpoint_at_insertion"))
        .abs()
        .alias("abs_distance_to_midpoint")
    ),
    # Relative distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    # / Midpoint at Insertion
    (
        (
            pl.col("price_at_insertion")
            .sub(pl.col("midpoint_at_insertion"))
            .abs()
        )
        .truediv(pl.col("midpoint_at_insertion"))
        .alias("rel_distance_to_midpoint")
    ),
    # TODO: Converted to microseconds
    # Log order lifetime
    # All order lifetimes need to be > 0, otherwise this would throw an error.
    # Further note: The durations in order_lifetime are implicitly casted
    # as Unix epoch nanoseconds and then the logarithm is calculated.
    # I've previously decided to lose a slight amount of accuracy by counting
    # microseconds but then the logarithm is computed against lower magnitudes.
    # At the same time, I'm also repeating the the same logarith but with bins
    # to prepare for 3.2.
    (
        pl.col("order_lifetime")
            .log1p()
            .alias("log_order_lifetime")
    ),
    (
        pl.col("order_lifetime")
            .log1p()
            .cut(
                breaks= log_order_lifetime_bins,
                labels=log_order_lifetime_bin_labels
            )
            .cast(pl.Utf8)
            .alias("log_order_lifetime_bins")          
    ),
    # Removal mechanism:
    # The execution size here is the final (cumulative) execution size.
    # Furthermore, we need to take into account the cases, in which an order has had its size modified (regardless of any associated trades).
    # over its lifecycle. I do this by:
    # 1. Adding all (positive) size_diff's for non-executed orders, but including the INSERT order and excluding the REMOVE order,
    # 2. Determining the difference between the size tally from 1. and the final execution size.
    # Thus, if
    #   execution_size = 0 (or size tally - execution_size = size tally), then none of the order has been filled and it has been cancelled,
    #   size tally - execution_size = 0 (or size tally = execution_size), then the entire order has been filled and then removed,
    #   size tally - execution_size > 0 (or size tally > execution_size), then the order has been partially filled, but still cancelled.
    (
       pl.when(pl.col("execution_size") == 0) \
            .then(pl.lit("Cancel")) \
            .when(pl.col("execution_size")==pl.col("size_diff_tally")) \
            .then(pl.lit("Trade (full)")) \
            .when(pl.col("execution_size").is_between(0, pl.col("size_diff_tally"), closed="none")) \
            .then(pl.lit("Trade (partial)")) \
            .otherwise(pl.lit("Unknown"))
        .alias("removal_mechanism")
    ),
])

In [281]:
# Sanity check: No order should have an order lifetime <= 0,
# which would mess up the logarithm.
quotes_collapsed \
    .filter(pl.col("order_lifetime") <= 0) \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,duration[ns],f64,f64,f64,f64,str


In [327]:
# Sanity check: No order's removal mechanism should be classified as "Unknown"
quotes_collapsed \
    .filter(pl.col("removal_mechanism")=="Unknown") \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,duration[ns],f64,f64,f64,f64,str


In [412]:
# Result
quotes_collapsed.collect().show(limit=10)
quotes_collapsed.collect().describe()

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str
50137,"""AQEU""",2023-09-01 07:00:23.832485,2023-09-01 11:00:00.100847,2023-09-01 11:00:00.100847,0,8.1,947,8.1,947,0,1,0.0,8.1,[],[],"""CONTINUOUS_TRADING""",947,14376268362,4.05,4.05,1.0,23.388845,"""22-24""","""Cancel"""
50138,"""AQEU""",2023-09-01 07:00:23.832985,2023-09-01 11:00:00.100871,2023-09-01 11:00:00.100871,0,5.0,757,5.0,757,0,1,5.0,8.1,[],[],"""CONTINUOUS_TRADING""",757,14376267886,6.55,1.55,0.236641,23.388845,"""22-24""","""Cancel"""
51441,"""AQEU""",2023-09-01 07:00:24.633269,2023-09-01 07:00:24.642583,2023-09-01 07:00:24.642583,0,7.104,750,7.104,750,0,1,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,9314,7.602,0.498,0.065509,9.139381,"""8-10""","""Cancel"""
51442,"""AQEU""",2023-09-01 07:00:24.633271,2023-09-01 07:00:24.633286,2023-09-01 07:00:24.633286,0,7.102,750,7.102,750,0,2,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,15,7.602,0.5,0.065772,2.772589,"""2-4""","""Cancel"""
51443,"""AQEU""",2023-09-01 07:00:24.633283,2023-09-01 07:00:24.642585,2023-09-01 07:00:24.642585,0,7.104,750,7.104,750,0,1,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,9302,7.602,0.498,0.065509,9.138092,"""8-10""","""Cancel"""
60099,"""AQEU""",2023-09-01 07:00:30.221642,2023-09-01 07:00:30.228353,2023-09-01 07:00:30.228353,0,7.112,750,7.112,750,0,1,7.112,8.1,[],[],"""CONTINUOUS_TRADING""",750,6711,7.606,0.494,0.064949,8.811652,"""8-10""","""Cancel"""
60135,"""AQEU""",2023-09-01 07:00:30.228330,2023-09-01 07:00:30.328332,2023-09-01 07:00:30.328332,0,7.114,1,7.114,1,0,1,7.114,8.1,[],[],"""CONTINUOUS_TRADING""",1,100002,7.607,0.493,0.064809,11.512955,"""10-12""","""Cancel"""
60143,"""AQEU""",2023-09-01 07:00:30.230594,2023-09-01 07:00:30.376114,2023-09-01 07:00:30.376114,0,7.116,750,7.116,750,0,1,7.116,8.1,[],[],"""CONTINUOUS_TRADING""",750,145520,7.608,0.492,0.064669,11.888076,"""10-12""","""Cancel"""
66381,"""AQEU""",2023-09-01 07:00:35.319647,2023-09-01 07:32:24.281250,2023-09-01 07:32:24.281250,0,7.18,203,7.18,203,0,1,5.0,7.18,[],[],"""CONTINUOUS_TRADING""",203,1908961603,6.09,1.09,0.178982,21.369825,"""20-22""","""Cancel"""


statistic,original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str
"""count""",189449.0,"""189449""","""189449""","""189449""","""189449""",189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189438.0,189449.0,189449.0,"""189449""",189449.0,189449.0,189438.0,189438.0,189438.0,189449.0,"""189449""","""189449"""
"""null_count""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0,"""0""",0.0,0.0,11.0,11.0,11.0,0.0,"""0""","""0"""
"""mean""",1.1159e18,null,"""2023-09-01 10:53:25.881948""","""2023-09-01 10:55:34.293706""","""2023-09-01 10:56:33.331417""",0.274121,7.312965,610.24908,7.313094,594.239722,23.358566,5.146852,7.298362,7.312898,null,null,null,607.826951,1.2841e8,7.305635,0.031021,0.004357,14.193243,null,null
"""std""",6.7317e17,null,null,null,null,7.522872,0.393784,989.119931,0.393746,950.20554,316.824017,10.091634,0.144286,0.085213,null,null,null,1176.221348,1.0115e9,0.105524,0.37751,0.053253,3.878808,null,null
"""min""",50137.0,"""AQEU""","""2023-09-01 05:30:04.652687""","""2023-09-01 07:00:12.541504""","""2023-09-01 07:00:12.541504""",0.0,0.001,1.0,0.001,1.0,0.0,1.0,0.0,7.086,null,null,"""CLOSED""",1.0,2.0,4.05,0.001,0.000134,1.098612,"""0-2""","""Cancel"""
"""25%""",2.1376e17,null,"""2023-09-01 08:14:15.609704""","""2023-09-01 08:17:57.923187""","""2023-09-01 08:17:57.923187""",0.0,7.27,259.0,7.27,250.0,0.0,1.0,7.27,7.28,null,null,null,250.0,134419.0,7.275,0.003,0.000422,11.808725,null,null
"""50%""",1.0814e18,null,"""2023-09-01 10:40:01.397597""","""2023-09-01 10:46:27.364529""","""2023-09-01 10:46:27.364529""",0.0,7.318,379.0,7.318,378.0,0.0,2.0,7.312,7.32,null,null,null,378.0,3.341794e6,7.316,0.007,0.000941,15.022019,null,null
"""75%""",1.6936e18,null,"""2023-09-01 13:33:46.024652""","""2023-09-01 13:35:27.259075""","""2023-09-01 13:35:48.813363""",0.0,7.382,637.0,7.382,635.0,0.0,5.0,7.38,7.386,null,null,null,637.0,2.2385409e7,7.383,0.019,0.00267,16.92392,null,null
"""max""",1.6936e18,"""XMIL""","""2023-09-01 15:30:00.610934""","""2023-09-01 15:40:00.008912""","""2023-09-01 15:40:00.008912""",1522.0,70.0,80000.0,70.0,80000.0,35004.0,247.0,7.44,8.1,null,null,"""POST_TRADE""",160000.0,3.6595e10,7.692,62.883,8.835605,24.323187,"""8-10""","""Trade (partial)"""


# 3.2

In [395]:
# This also includes the orders that are not found in the LOB
alt.Chart(trades_eu.collect()).mark_bar().encode(
    x = alt.X('venue:N', axis=alt.Axis(labelAngle=0), title="Venues"),
    xOffset = 'trade_type:N',
    y=alt.Y('count():Q', title="No. of Trades"),
    color=alt.Color('trade_type:N', title="Trade type"),
)

alt.Chart(...)

In [380]:
quotes_collapsed \
    .group_by("venue") \
    .agg(
        [
            pl.col("venue")
                .count()
                .alias("total_number_of_orders"),
            pl.col("number_of_updates")
                .sum()
                .alias("total_number_of_updates"),
            pl.col("number_of_updates")
                .mean()
                .alias("avg_number_of_updates_per_order"),
            pl.col("order_lifetime")
                .mean()
                .cast(pl.Duration)
                .alias("avg_order_lifetime"),
            pl.col("order_lifetime")
                .median()
                .cast(pl.Duration)
                .alias("median_order_lifetime"),
            pl.col("execution_size")
                .sum()
                .alias("total_executed_size"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Cancel")
                .count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_cancels"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Trade (full)").count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_full_trades"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Trade (partial)").count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_partial_trades"),
            pl.col("rel_distance_to_midpoint")
                .mean()
                .alias("avg_relative_distance_to_midpoint"),
            pl.col("rel_distance_to_midpoint")
                .median()
                .alias("median_relative_distance_to_midpoint"),
        ]
    ) \
    .sort(by="venue") \
    .collect()

venue,total_number_of_orders,total_number_of_updates,avg_number_of_updates_per_order,avg_order_lifetime,median_order_lifetime,total_executed_size,rel_number_of_cancels,rel_number_of_full_trades,rel_number_of_partial_trades,avg_relative_distance_to_midpoint,median_relative_distance_to_midpoint
str,u32,u32,f64,duration[μs],duration[μs],i64,f64,f64,f64,f64,f64
"""AQEU""",37184,3099,0.083342,17s 137378µs,405132µs,98851,0.991502,0.006912,0.001587,0.004273,0.000823
"""CEUX""",49581,11113,0.224138,1m 4s 440993µs,5s 118348µs,739598,0.952744,0.04421,0.003046,0.00396,0.001893
"""TQEX""",10884,5642,0.518376,47s 896102µs,1s 826191µs,94742,0.972253,0.024531,0.003216,0.00432,0.001495
"""XETR""",85575,15930,0.186152,3m 43s 408205µs,5s 363332µs,3453737,0.951072,0.044663,0.004265,0.004667,0.000687
"""XMIL""",6225,16148,2.594056,2m 17s 467553µs,6s 525311µs,38329,0.994538,0.004016,0.001446,0.003184,0.002674


There is considerable right skewness in the order lifetimes across all venues, meaning most orders live briefly, but some very persistent
orders inflate the average. Similarly, though not as pronounced, this applies to the rel. distance to the midpoint.

#### AQEU
- Relatively close to the midpoint for most orders, but some traders distort the average by placing (passive) orders far away in the book
- The shortest average (and median) order lifetimes by far, most orders disappear almost immediately (cancelled)
- Could be indicative of HFT & quote stuffing


#### CEUX
- Moderate order lifetimes overall, moderate update frequence, moderate rel. distance to midpoint on median (but the opposite on avg.), moderate-to-high execution share
- High rate of trades that are filled fully.
- CEUX seems to have the most dark trades over all venues by far. Generally, this tends to fragment price discovery. Not unheard of in an MTF.


#### TQEX
- Most orders go through very fast and most are not inserted as close as they can be to the midpoint.
- Releatively low update frequency
- Doesn't seem that aggressive of a market


#### XETR
- Highest execution share by far
- Most orders are very close to the midpoint but, also, some quote very far from the midpoint and exist for relatively long time, could be large (passive) institutional orders
- It's Xetra, it's a regulated market


#### XMIL
- Quite high update frequency at very high cancellation rates. Could indicate active management.
- Order are generally quoted relatively far from the midpoint and they have relatively long lifetimes.
- 

In [466]:
# Ideally, the interactive tooltip will work when hovered over, which will show
# relevant information if a bar is not easy to see.
alt.Chart(quotes_collapsed.collect(), title="Log order life times for entire LOB per venue").mark_bar().encode(
    x=alt.X(
        "log_order_lifetime_bins:N",
        axis=alt.Axis(labelAngle=0),
        title="Log order lifetimes",
    ),
    xOffset="venue:N",
    y=alt.Y(
        "count():Q",
        title="No. of Trades",
    ),
    color=alt.Color(
        "venue:N",
        title="Venue",
    ),
    tooltip=[
        alt.Tooltip("log_order_lifetime_bins:N", title="Log order lifetime"),
        alt.Tooltip("venue:N", title="Venue"),
        alt.Tooltip("count():Q", title="No. of Trades"),
    ],
)

alt.Chart(...)

# 3.3

### Testing area


In [209]:
quotes_inc_eu_test_filter = pl.col("trade_id").is_in([42943562011806, 118])
quotes_inc_eu \
    .filter(quotes_inc_eu_test_filter).sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]).collect()


side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str


In [136]:
quotes_collapsed.filter(pl.col("removal_mechanism")=="Trade (partial)").collect().sample(10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,duration[ns],f64,f64,f64,f64,str
1081350376586134041,"""CEUX""",2023-09-01 09:15:27.547771,2023-09-01 09:15:27.662061,2023-09-01 09:15:27.662061,1,7.304,291,7.304,223,68,1,7.3,7.304,"[{2023-09-01 09:15:27.548701,7.304,223}]",[{6269133180374}],"""CONTINUOUS_TRADING""",291,114290µs,7.302,0.002,0.000274,18.55425,"""Trade (partial)"""
1693570190657109475,"""XETR""",2023-09-01 12:09:50.657120234,2023-09-01 12:09:51.814099611,2023-09-01 12:09:51.814099611,1,7.338,379,7.338,92,287,1,7.334,7.338,"[{2023-09-01 12:09:51.813148186,7.338,92}]",[{1693570191813124427}],"""CONTINUOUS_TRADING""",379,1s 156979377ns,7.336,0.002,0.000273,20.869078,"""Trade (partial)"""
23122422,"""AQEU""",2023-09-01 15:07:35.787303,2023-09-01 15:08:47.931350,2023-09-01 15:08:47.931350,1,7.382,363,7.382,77,286,1,7.382,7.388,"[{2023-09-01 15:08:47.930225,7.382,77}]",[{185727}],"""CONTINUOUS_TRADING""",363,1m 12s 144047µs,7.385,0.003,0.000406,25.001931,"""Trade (partial)"""
1693582164193433841,"""XETR""",2023-09-01 15:29:24.193442537,2023-09-01 15:29:40.833359893,2023-09-01 15:29:40.833359893,1,7.364,636,7.364,246,390,1,7.364,7.366,"[{2023-09-01 15:29:40.833283786,7.364,246}]",[{1693582180833260728}],"""CONTINUOUS_TRADING""",636,16s 639917356ns,7.365,0.001,0.000136,23.53507,"""Trade (partial)"""
1693565580878757219,"""XETR""",2023-09-01 10:53:00.878769790,2023-09-01 11:00:00.096401017,2023-09-01 11:00:00.096401017,1,7.286,379,7.286,79,300,1,7.286,7.292,"[{2023-09-01 10:53:24.102909180,7.286,79}]",[{1693565604102861116}],"""INTRADAY_AUCTION""",379,6m 59s 217631227ns,7.289,0.003,0.000412,26.761656,"""Trade (partial)"""
1693563293095382054,"""XETR""",2023-09-01 10:14:53.095400078,2023-09-01 11:00:00.096401017,2023-09-01 12:38:30.733873868,2,7.352,900,7.352,900,900,14,7.32,7.326,"[{2023-09-01 12:38:30.733778948,7.352,497}, {2023-09-01 12:38:30.733829900,7.352,31}]","[{1693571910733758327}, {1693571910733804844}, {1693571910733853125}]","""INTRADAY_AUCTION""",1800,45m 7s 1000939ns,7.323,0.029,0.00396,28.626862,"""Trade (partial)"""
1081350376586134031,"""CEUX""",2023-09-01 09:15:27.545164,2023-09-01 09:15:28.506132,2023-09-01 09:15:28.506132,1,7.306,440,7.306,425,15,1,7.3,7.306,"[{2023-09-01 09:15:28.506030,7.306,425}]",[{6269133180378}],"""CONTINUOUS_TRADING""",440,960968µs,7.303,0.003,0.000411,20.683452,"""Trade (partial)"""
1693566394042959286,"""XETR""",2023-09-01 11:06:34.042968571,2023-09-01 11:06:42.682396778,2023-09-01 11:06:42.682396778,1,7.286,812,7.286,304,508,1,7.286,7.29,"[{2023-09-01 11:06:42.642273569,7.286,304}]",[{1693566402642234968}],"""CONTINUOUS_TRADING""",812,8s 639428207ns,7.288,0.002,0.000274,22.879602,"""Trade (partial)"""
1693554287252934974,"""XETR""",2023-09-01 07:44:47.252945513,2023-09-01 11:00:00.096401017,2023-09-01 11:52:21.042361548,0,7.34,1005,7.34,1005,1005,28,7.264,7.27,[],[{1693569141042324486}],"""INTRADAY_AUCTION""",2010,3h 15m 12s 843455504ns,7.267,0.073,0.010045,30.091707,"""Trade (partial)"""


In [348]:
test_ids = [50137] # [1081350376584932672, 1428746]
test_filter = pl.col("original_order_id").is_in(test_ids)

quotes_inc_eu \
    .filter(test_filter) \
    .sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .collect() \
    .show(limit=15)

quotes_collapsed \
    .filter((pl.col("best_bid_price_at_insertion").is_null()) | (pl.col("best_ask_price_at_insertion").is_null())) \
    .collect() \
    # .show(limit=10)

# test_filter3 = pl.col("size_at_insertion") < pl.col("execution_size") # pl.col("size_before_removal").is_between(pl.col("size_at_insertion"), pl.col("execution_size"), closed="none")
# quotes_collapsed \
#     .filter(test_filter3) \
#     .collect() \
#     .show(limit=10)

side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue,size_diff,order_order,size_diff_cum_sum
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str,i64,u32,i64
"""ASK""",8.1,947,50137,2023-09-01 07:00:23.832485,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,null,0,0,true,true,50137,null,0,0,1,0,1,947,1,0,"""CONTINUOUS_TRADING""","""AQEU""",947,1,0
"""ASK""",null,0,null,2023-09-01 11:00:00.100847,"""REMOVE""",8.1,947,50137,false,null,0,0,10,7.304,750,6691,7.272,596,3742,false,false,50137,null,6691,11,1,1,0,0,11,7,"""CONTINUOUS_TRADING""","""AQEU""",-947,2,-947


original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str
50137,"""AQEU""",2023-09-01 07:00:23.832485,2023-09-01 11:00:00.100847,2023-09-01 11:00:00.100847,0,8.1,947,8.1,947,0,1,null,8.1,[],[],"""CONTINUOUS_TRADING""",947,14376268362,null,null,null,23.388845,"""Cancel"""
9104176,"""AQEU""",2023-09-01 11:00:01.350086,2023-09-01 11:00:06.349159,2023-09-01 11:00:06.349159,0,7.064,2800,7.064,2800,0,1,7.064,null,[],[],"""CONTINUOUS_TRADING""",2800,4999073,null,null,null,15.424763,"""Cancel"""
9126333,"""AQEU""",2023-09-01 11:00:33.804366,2023-09-01 11:01:57.581379,2023-09-01 11:01:57.581379,0,7.062,2800,7.062,2800,0,1,7.062,null,[],[],"""CONTINUOUS_TRADING""",2800,83777013,null,null,null,18.243669,"""Cancel"""
9184912,"""AQEU""",2023-09-01 11:02:17.361765,2023-09-01 11:03:22.573671,2023-09-01 11:03:22.573671,0,7.282,750,7.282,750,0,1,7.282,null,[],[],"""CONTINUOUS_TRADING""",750,65211906,null,null,null,17.993153,"""Cancel"""
9184913,"""AQEU""",2023-09-01 11:02:17.361769,2023-09-01 11:32:47.589726,2023-09-01 11:32:47.589726,0,7.218,728,7.218,728,0,2,7.282,null,[],[],"""CONTINUOUS_TRADING""",728,1830227957,null,null,null,21.327706,"""Cancel"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1081350376584637819,"""CEUX""",2023-09-01 07:00:00.023784,2023-09-01 07:00:23.824332,2023-09-01 07:00:23.824332,0,7.074,1828,7.074,1828,0,1,7.074,null,[],[],"""CONTINUOUS_TRADING""",1828,23800548,null,null,null,16.985219,"""Cancel"""
1693546204652687295,"""XMIL""",2023-09-01 05:30:04.652687295,2023-09-01 15:40:00.008912129,2023-09-01 15:40:00.008912129,0,6.98,65,6.98,65,0,1,6.98,null,[],[],"""CLOSED""",65,36595356224,null,null,null,24.323187,"""Cancel"""
1693551595638614505,"""XETR""",2023-09-01 07:00:23.819462822,2023-09-01 07:00:23.827965471,2023-09-01 07:00:23.827965471,0,7.114,368,7.114,368,0,1,7.114,null,[],[],"""CONTINUOUS_TRADING""",368,8502,null,null,null,9.048174,"""Cancel"""


In [142]:
show_histogram(
    df=quotes_collapsed, 
    column_name="removal_mechanism",
    x_label="Removal mechanism"
)

alt.LayerChart(...)